## Mimic Guitar Amplifier EQ Gain Controls Using Nodal Analysis and State-space Equations

### Imports and Audio Loading

In [31]:
import numpy as np
from scipy.io.wavfile import read
from scipy.signal import lfilter
import IPython.display as ipd

In [32]:
rate, input_audio = read("../clips/mine/fairest_of_the_seasons.wav")
# Convert to mono if stereo
if input_audio.ndim > 1:
    input_audio = input_audio.mean(axis=1).astype(input_audio.dtype)

# Convert to floating point and normalize
input_audio = input_audio.astype(np.float32) / np.max(np.abs(input_audio))

# Trim to same length
dur = 10
N_samples = dur * rate
input_audio = input_audio[:N_samples]

/var/folders/yz/xcf27lrx69l4v_mqmrv7_z_80000gn/T/ipykernel_27164/2433788524.py:1: WavFileWarning: Chunk (non-data) not understood, skipping it.
  rate, input_audio = read("../clips/mine/fairest_of_the_seasons.wav")


### Set up State-space Equations

In [70]:
treble_pot = 0.01
mid_pot = 0.01
bass_pot = 1

In [71]:
C1 = 250e-12 # F, treble cap
C2 = 100e-9 # F, bass cap
C3 = 47e-9 # F, mid cap

Rs = 38e3 # Ω, source
RL = 1e6 # Ω, load

Rt = 250e3 # Ω, treble pot
RM = 10e3 # Ω, mid pot
RB = 1e6 # Ω, bass pot

R1 = Rt * treble_pot
R2 = RM * mid_pot
R3 = RB * bass_pot

In [72]:
g1 = 1/R1
g2 = 1/R2
g3 = 1/R3
gs = 1/Rs

K = 2*rate

a0 = gs*g2*g3 + g1*g2*g3
a1 = gs*g2*(C1+C2) + gs*g3*(C2+C3) + g2*g3*(C1+C3) + g1*g2*(C1+C2)
a2 = gs*C1*C2 + g2*C1*C3 + g3*C2*C3 + g1*C1*C2

b0_d = (gs*C1*C2*K**2 + K*(gs*g2*(C1+C2) + gs*g3*C2) + gs*g2*g3) / (a2*K**2 + a1*K + a0)
b1_d = (-2*gs*C1*C2*K**2 + 2*gs*g2*g3) / (a2*K**2 + a1*K + a0)
b2_d = (gs*C1*C2*K**2 - K*(gs*g2*(C1+C2) + gs*g3*C2) + gs*g2*g3) / (a2*K**2 + a1*K + a0)

a1_d = (-2*a2*K**2 + 2*a0) / (a2*K**2 + a1*K + a0)
a2_d = (a2*K**2 - a1*K + a0) / (a2*K**2 + a1*K + a0)

In [73]:
output_audio = lfilter([b0_d, b1_d, b2_d], [1, a1_d, a2_d], input_audio)
ipd.display(ipd.Audio(output_audio, rate=rate, normalize=False))